# Model Selection and Encoding Experiments
In this notebook we evaluate different machine learning models and encoding strategies for the rent price prediction task.

## Imports and Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

%load_ext autoreload
%autoreload all
from src.models.pipeline_utils import evaluate_models

## Data Preparation

In [ ]:
data = pd.read_csv('../data/processed/train_fe.csv')
black_list = ['id', 'price', 'price_bin', 'residential',
              'neighborhood', 'description', 'detail']

feats = [col for col in data.columns if col not in black_list]

num_feats = [
    col for col in data.select_dtypes(exclude='object').columns if col not in black_list
]
cat_feats = [
    col for col in data.select_dtypes(include='object').columns if col not in black_list
]

X = data[feats]
y = data['price']

## Model Configurations

In [3]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmsle',
    'seed': 7,
    'tree_method': 'hist',
    'grow_policy': 'lossguide',
}

lgb_params = {
    'random_state': 7,
    'verbose': -1,
    'force_col_wise': True,
}

MODELS = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=7),
    'Lasso': Lasso(random_state=7),
    'ElasticNet': ElasticNet(random_state=7),
    'KNN': KNeighborsRegressor(),
    'SVR': SVR(),
    'Decision Tree': DecisionTreeRegressor(random_state=7),
    'Extra Trees': ExtraTreesRegressor(random_state=7, n_jobs=-1),
    'Random Forest': RandomForestRegressor(random_state=7, n_jobs=-1),
    'XGBoost': XGBRegressor(**xgb_params),
    'LightGBM': LGBMRegressor(**lgb_params)
}

## Numerical Features Only

In [4]:
results_dict = {}

In [5]:
pipeline_config = {'numerical only': {
    'numerical_features': num_feats,
    'encoding_strategy': 'none'
}}

df_results = evaluate_models(MODELS, X[num_feats], y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.3095, val_rmsle: 0.3129
[Fold 1] train_rmsle: 0.3090, val_rmsle: 0.3146
[Fold 2] train_rmsle: 0.3082, val_rmsle: 0.3167
RMSLE: 0.3147 ± 0.0016

Ridge
[Fold 0] train_rmsle: 0.3095, val_rmsle: 0.3129
[Fold 1] train_rmsle: 0.3090, val_rmsle: 0.3148
[Fold 2] train_rmsle: 0.3082, val_rmsle: 0.3167
RMSLE: 0.3148 ± 0.0015

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3120, val_rmsle: 0.3874
[Fold 1] train_rmsle: 0.3164, val_rmsle: 0.3786
[Fold 2] train_rmsle: 0.3104, val_rmsle: 0.3909
RMSLE: 0.3856 ± 0.0052

SVR
[Fold 0] train_rmsle: 0.1709, val_rmsle: 0.2919
[Fold 1] train_rmsle: 0.1721, val_rmsle: 0.3078
[Fold 2] train

In [6]:
df_results

,model,RMSLE
10,numerical only LightGBM,0.2497
9,numerical only XGBoost,0.2526
8,numerical only Random Forest,0.2627
7,numerical only Extra Trees,0.2628
5,numerical only SVR,0.3003
0,numerical only Linear Regression,0.3147
1,numerical only Ridge,0.3148
6,numerical only Decision Tree,0.3758
4,numerical only KNN,0.3856
2,numerical only Lasso,0.7359


**Compare with baseline:**
- Boosting models improved significantly after feature engineering:
    - **XGBoost:** 0.286 → 0.253
    - **LightGBM:** 0.290 → 0.250
- Tree ensembles also improved:
    - **Random Forest:** 0.293 → 0.263
    - **Extra Trees:** 0.301 → 0.263
- Linear models improved dramatically:
    - **Linear Regression:** 0.485 → 0.315
    - **Ridge:** 0.315
- **KNN**, **Decision Tree**, **Lasso**, and **ElasticNet** consistently underperformed (RMSLE > 0.315) and are not recommended for further tuning.

Given these results, future experiments should concentrate on LightGBM and XGBoost as primary models, with Ridge, Linear Regression, and SVR serving as complementary baselines. Additional experiments with categorical feature encoding (one-hot, ordinal, and combined approaches) are expected to further enhance the performance of boosting and linear models.



## Encoding Experiments

### OneHot Encoding

In [7]:
pipeline_config = {'onehot encoding': {
    'encoding_strategy': 'onehot',
    'numerical_features': num_feats,
    'categorical_features': cat_feats
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2694, val_rmsle: 0.2737
[Fold 1] train_rmsle: 0.2661, val_rmsle: 0.2810
[Fold 2] train_rmsle: 0.2680, val_rmsle: 0.2776
RMSLE: 0.2774 ± 0.0030

Ridge
[Fold 0] train_rmsle: 0.2694, val_rmsle: 0.2736
[Fold 1] train_rmsle: 0.2661, val_rmsle: 0.2808
[Fold 2] train_rmsle: 0.2681, val_rmsle: 0.2776
RMSLE: 0.2773 ± 0.0030

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3014, val_rmsle: 0.3717
[Fold 1] train_rmsle: 0.3025, val_rmsle: 0.3658
[Fold 2] train_rmsle: 0.2967, val_rmsle: 0.3754
RMSLE: 0.3709 ± 0.0039

SVR
[Fold 0] train_rmsle: 0.1490, val_rmsle: 0.2715
[Fold 1] train_rmsle: 0.1463, val_rmsle: 0.2899
[Fold 2] train

In [8]:
df_results

,model,RMSLE
21,onehot encoding LightGBM,0.2474
20,onehot encoding XGBoost,0.2487
10,numerical only LightGBM,0.2497
9,numerical only XGBoost,0.2526
18,onehot encoding Extra Trees,0.2544
19,onehot encoding Random Forest,0.2590
8,numerical only Random Forest,0.2627
7,numerical only Extra Trees,0.2628
12,onehot encoding Ridge,0.2773
11,onehot encoding Linear Regression,0.2774


One-hot encodong for categorical features increase performace for all models.

### Ordinal Encoding

In [9]:
pipeline_config = {'ordinal encoding': {
    'encoding_strategy': 'ordinal',
    'numerical_features': num_feats,
    'categorical_features': cat_feats
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2953, val_rmsle: 0.2984
[Fold 1] train_rmsle: 0.2932, val_rmsle: 0.3034
[Fold 2] train_rmsle: 0.2949, val_rmsle: 0.3003
RMSLE: 0.3007 ± 0.0021

Ridge
[Fold 0] train_rmsle: 0.2953, val_rmsle: 0.2983
[Fold 1] train_rmsle: 0.2932, val_rmsle: 0.3034
[Fold 2] train_rmsle: 0.2949, val_rmsle: 0.3003
RMSLE: 0.3007 ± 0.0021

Lasso
[Fold 0] train_rmsle: 0.7014, val_rmsle: 0.6953
[Fold 1] train_rmsle: 0.6995, val_rmsle: 0.7005
[Fold 2] train_rmsle: 0.6978, val_rmsle: 0.7043
RMSLE: 0.7000 ± 0.0037

ElasticNet
[Fold 0] train_rmsle: 0.6754, val_rmsle: 0.6698
[Fold 1] train_rmsle: 0.6735, val_rmsle: 0.6750
[Fold 2] train_rmsle: 0.6726, val_rmsle: 0.6779
RMSLE: 0.6742 ± 0.0034

KNN
[Fold 0] train_rmsle: 0.3019, val_rmsle: 0.3877
[Fold 1] train_rmsle: 0.3046, val_rmsle: 0.3739
[Fold 2] train_rmsle: 0.3061, val_rmsle: 0.3720
RMSLE: 0.3779 ± 0.0070

SVR
[Fold 0] train_rmsle: 0.2527, val_rmsle: 0.2741
[Fold 1] train_rmsle: 0.2498, val_rmsle: 0.2803
[Fold 2] train

In [10]:
df_results

,model,RMSLE
32,ordinal encoding LightGBM,0.2468
21,onehot encoding LightGBM,0.2474
20,onehot encoding XGBoost,0.2487
10,numerical only LightGBM,0.2497
31,ordinal encoding XGBoost,0.2515
9,numerical only XGBoost,0.2526
18,onehot encoding Extra Trees,0.2544
29,ordinal encoding Extra Trees,0.2559
30,ordinal encoding Random Forest,0.2586
19,onehot encoding Random Forest,0.2590


- Gradient boosting model **LightGBM** achieved the best performance with **RMSLE=0.247**, outperforming linear, tree-based, and kernel methods. 
- The choice between one-hot and ordinal encoding had little impact on boosting and tree-based models, while  for Linear Regression and Ridge one-hot encoding performed better.

### Combine Ordinal and OneHot Encoding

In [11]:
cat_feats

['subway',
 'district',
 'floor_binned',
 'num_storeys_binned',
 'residential_top',
 'neighborhood_top',
 'district_price_tier',
 'subway_line']

In [12]:
pipeline_config = {'mixed encoding': {
    'encoding_strategy': 'mixed',
    'numerical_features': num_feats,
    'categorical_features': cat_feats,
    'onehot_features': [
        'subway', 'district', 'residential_top',
        'neighborhood_top', 'subway_line'
    ],
    'ordinal_categories': {
        'floor_binned': ['1', '2-5', '6-10', '11-15', '16-20', '21-25', '26+'],
        'num_storeys_binned': ['1-5', '6-10', '11-15', '16-20', '21-30', '31+'],
        'district_price_tier': ['tier_1', 'tier_2', 'tier_3']
    }
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2696, val_rmsle: 0.2739
[Fold 1] train_rmsle: 0.2666, val_rmsle: 0.2805
[Fold 2] train_rmsle: 0.2682, val_rmsle: 0.2778
RMSLE: 0.2774 ± 0.0027

Ridge
[Fold 0] train_rmsle: 0.2697, val_rmsle: 0.2737
[Fold 1] train_rmsle: 0.2667, val_rmsle: 0.2803
[Fold 2] train_rmsle: 0.2682, val_rmsle: 0.2778
RMSLE: 0.2773 ± 0.0027

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3010, val_rmsle: 0.3706
[Fold 1] train_rmsle: 0.3015, val_rmsle: 0.3677
[Fold 2] train_rmsle: 0.2964, val_rmsle: 0.3717
RMSLE: 0.3700 ± 0.0017

SVR
[Fold 0] train_rmsle: 0.1591, val_rmsle: 0.2682
[Fold 1] train_rmsle: 0.1574, val_rmsle: 0.2838
[Fold 2] train

In [13]:
df_results

,model,RMSLE
32,ordinal encoding LightGBM,0.2468
21,onehot encoding LightGBM,0.2474
43,mixed encoding LightGBM,0.2480
20,onehot encoding XGBoost,0.2487
10,numerical only LightGBM,0.2497
31,ordinal encoding XGBoost,0.2515
42,mixed encoding XGBoost,0.2521
9,numerical only XGBoost,0.2526
40,mixed encoding Extra Trees,0.2528
18,onehot encoding Extra Trees,0.2544


The experiments confirm that **boosting models (LightGBM, XGBoost)** consistently deliver the best performance across all encoding strategies.
- **LightGBM** achieved the lowest RMSLE (**0.2468**) with **ordinal encoding**, slightly outperforming one-hot (**0.2474**) and mixed encoding (**0.2480**). Using only numerical features resulted in slightly worse performance, showing the benefit of categorical feature encoding.
- **XGBoost** showed similar patterns but with slightly higher RMSLE scores(the best is **0.2487** with one-hot encoding).
- **Extra Trees** achieved the lowest RMSLE of **0.2528** with mixed encoding, that slightly behind XGBoost with only numerical features.
- **Tree-based models (LightGBM, XGBoost, Extra Trees, Random Forest)** were clearly superior to **linear models** (Ridge, Linear Regression) and **non-tree models** (SVR, KNN), with a noticeable performance gap.
- **Random Forest** achieved moderate results (RMSLE around **0.259**) with no meaningful difference betweens all encoding strategies but still better than using only numerical features. 
- **Linear models (Ridge, Linear Regression)** and **SVR** stabilized around **0.277**, performing reasonably well but not competitive with boosting and ensemble models.
- Simpler models **(Decision Tree, KNN)** underperform (**0.371**), indicating poor suitability for this task.
- Regularized linear models **(Lasso, ElasticNet)** are the weakest, underperformed regardless of encoding strategy, suggesting they are not competitive for this prediction task.


**Conclusion**:
- The best-performing and most stable approach is **LightGBM** with ordinal or one-hot encoding, achieving RMSLE = **0.2468**. Further tuning and feature engineering should prioritize boosting models, as encoding choice does not significantly affect their performance, while other models can serve as comparative baselines only.